# EDA del Titanic con Polars

Cuaderno instructivo para explorar el dataset original del Titanic sin transformar sus datos. Trabajaremos con conteos y tasas de supervivencia por grupos. **Asociación no implica causalidad**: las diferencias observadas describen este conjunto de datos y no demuestran que una variable cause la supervivencia.

No se realizan limpieza, imputación ni eliminación de filas. Tampoco se incluyen gráficos, modelos de machine learning ni benchmarks.

## 1. Carga robusta y comprobaciones iniciales

La ruta esperada es `../05-recursos/datasets/titanic.csv`, relativa a este cuaderno. Como Jupyter puede arrancar desde otro directorio, probamos esa ruta desde el directorio de trabajo y también desde la ubicación del archivo cuando está disponible.

In [1]:
from pathlib import Path
from hashlib import sha256
import polars as pl

# Ruta del recurso, manteniendo explícitamente la ruta relativa del proyecto.
relative_dataset = Path("../05-recursos/datasets/titanic.csv")
candidates = [Path('titanic.csv'), Path.cwd() / relative_dataset]
# En Jupyter, __file__ no suele existir; esta alternativa facilita ejecutar el cuaderno desde otro cwd.
try:
    notebook_dir = Path(__file__).resolve().parent
    candidates.append(notebook_dir / relative_dataset)
except NameError:
    pass
# También localizamos la raíz si el kernel se lanzó desde cualquier subdirectorio del proyecto.
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(parent / "02-tratamiento-datos" / "05-recursos" / "datasets" / "titanic.csv")

dataset_path = next((path.resolve() for path in candidates if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(f"No se encontró el dataset en la ruta esperada: {relative_dataset}")

df = pl.read_csv(dataset_path)
print(f"CSV cargado desde: {dataset_path}")
HUELLA_CSV = sha256(dataset_path.read_bytes()).hexdigest()
print('SHA-256:', HUELLA_CSV)
print(f"Shape: {df.shape}")
print("Esquema:")
print(df.schema)
print("Nulos por columna:")
print(df.null_count())


CSV cargado desde: /home/claude/tit/titanic.csv
SHA-256: 8331e5a2532f5fdb66153a8fc053df08a7dc9d2584f246c7fb1752f0a7139ca5
Shape: (891, 12)
Esquema:
Schema({'PassengerId': Int64, 'Survived': Int64, 'Pclass': Int64, 'Name': String, 'Sex': String, 'Age': Float64, 'SibSp': Int64, 'Parch': Int64, 'Ticket': String, 'Fare': Float64, 'Cabin': String, 'Embarked': String})
Nulos por columna:
shape: (1, 12)
┌─────────────┬──────────┬────────┬──────┬───┬────────┬──────┬───────┬──────────┐
│ PassengerId ┆ Survived ┆ Pclass ┆ Name ┆ … ┆ Ticket ┆ Fare ┆ Cabin ┆ Embarked │
│ ---         ┆ ---      ┆ ---    ┆ ---  ┆   ┆ ---    ┆ ---  ┆ ---   ┆ ---      │
│ u32         ┆ u32      ┆ u32    ┆ u32  ┆   ┆ u32    ┆ u32  ┆ u32   ┆ u32      │
╞═════════════╪══════════╪════════╪══════╪═══╪════════╪══════╪═══════╪══════════╡
│ 0           ┆ 0        ┆ 0      ┆ 0    ┆ … ┆ 0      ┆ 0    ┆ 687   ┆ 2        │
└─────────────┴──────────┴────────┴──────┴───┴────────┴──────┴───────┴──────────┘


In [2]:
expected_columns = [
    "PassengerId", "Survived", "Pclass", "Name", "Sex", "Age",
    "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"
]
assert df.shape == (891, 12), df.shape
assert df.columns == expected_columns, df.columns
null_counts = df.null_count().row(0, named=True)
assert null_counts["Age"] == 177, null_counts
assert null_counts["Cabin"] == 687, null_counts
assert null_counts["Embarked"] == 2, null_counts
print("Comprobaciones de estructura y nulos: OK")


Comprobaciones de estructura y nulos: OK


## 2. Función para resumir supervivencia

En cada grupo mostramos `total` (pasajeros de ese grupo), `supervivientes` (suma de Survived) y `tasa = supervivientes / total`. No se repite el total global en las filas de los grupos. Las claves y el orden son comunes a los tres motores.

In [3]:
def resumen_supervivencia(tabla: pl.DataFrame, por: list[str]) -> pl.DataFrame:
    """Devuelve denominador del grupo, supervivientes y tasa."""
    return (
        tabla.group_by(por, maintain_order=True)
        .agg(
            total=pl.len(),
            supervivientes=pl.col("Survived").sum(),
        )
        .with_columns(
            (pl.col("supervivientes") / pl.col("total")).alias("tasa"),
        )
        .select(por + ["total", "supervivientes", "tasa"])
        .sort(por)
    )

global_eda = (
    df.select(
        pl.len().alias("total"),
        pl.col("Survived").sum().alias("supervivientes"),
    )
    .with_columns((pl.col("supervivientes") / pl.col("total")).alias("tasa"))
)
print(global_eda)
assert global_eda.item(0, "total") == 891
assert global_eda.item(0, "supervivientes") == 342
assert global_eda.columns == ['total', 'supervivientes', 'tasa']


shape: (1, 3)
┌───────┬────────────────┬──────────┐
│ total ┆ supervivientes ┆ tasa     │
│ ---   ┆ ---            ┆ ---      │
│ u32   ┆ i64            ┆ f64      │
╞═══════╪════════════════╪══════════╡
│ 891   ┆ 342            ┆ 0.383838 │
└───────┴────────────────┴──────────┘


## 3. Supervivencia por sexo

Comparamos los grupos `female` y `male`. La comparación identifica una asociación descriptiva en esta muestra; no permite afirmar causalidad.

In [4]:
por_sexo = resumen_supervivencia(
    df.filter(pl.col("Sex").is_in(["female", "male"])), ["Sex"]
)
print(por_sexo)
expected_sex = {"female": (314, 233), "male": (577, 109)}
for row in por_sexo.iter_rows(named=True):
    total, survived = expected_sex[row["Sex"]]
    assert (row["total"], row["supervivientes"]) == (total, survived)
    assert abs(row["tasa"] - survived / total) < 1e-6


shape: (2, 4)
┌────────┬───────┬────────────────┬──────────┐
│ Sex    ┆ total ┆ supervivientes ┆ tasa     │
│ ---    ┆ ---   ┆ ---            ┆ ---      │
│ str    ┆ u32   ┆ i64            ┆ f64      │
╞════════╪═══════╪════════════════╪══════════╡
│ female ┆ 314   ┆ 233            ┆ 0.742038 │
│ male   ┆ 577   ┆ 109            ┆ 0.188908 │
└────────┴───────┴────────────────┴──────────┘


## 4. Supervivencia por clase (`Pclass`)

In [5]:
por_clase = resumen_supervivencia(df.filter(pl.col("Pclass").is_in([1, 2, 3])), ["Pclass"]).sort("Pclass")
print(por_clase)
expected_class = {1: (216, 136), 2: (184, 87), 3: (491, 119)}
for row in por_clase.iter_rows(named=True):
    total, survived = expected_class[row["Pclass"]]
    assert (row["total"], row["supervivientes"]) == (total, survived)
    assert abs(row["tasa"] - survived / total) < 1e-6


shape: (3, 4)
┌────────┬───────┬────────────────┬──────────┐
│ Pclass ┆ total ┆ supervivientes ┆ tasa     │
│ ---    ┆ ---   ┆ ---            ┆ ---      │
│ i64    ┆ u32   ┆ i64            ┆ f64      │
╞════════╪═══════╪════════════════╪══════════╡
│ 1      ┆ 216   ┆ 136            ┆ 0.62963  │
│ 2      ┆ 184   ┆ 87             ┆ 0.472826 │
│ 3      ┆ 491   ┆ 119            ┆ 0.242363 │
└────────┴───────┴────────────────┴──────────┘


## 5. Cruce de clase y sexo

Ordenamos primero `Pclass` ascendente y después `Sex` en el orden didáctico `female`, `male`.

In [6]:
por_clase_sexo = (
    resumen_supervivencia(df.filter(pl.col("Pclass").is_in([1, 2, 3]) & pl.col("Sex").is_in(["female", "male"])), ["Pclass", "Sex"])
    .with_columns(pl.col("Sex").cast(pl.Enum(["female", "male"])))
    .sort(["Pclass", "Sex"])
)
print(por_clase_sexo)
expected_combo = {
    (1, "female"): (94, 91), (1, "male"): (122, 45),
    (2, "female"): (76, 70), (2, "male"): (108, 17),
    (3, "female"): (144, 72), (3, "male"): (347, 47),
}
for row in por_clase_sexo.iter_rows(named=True):
    total, survived = expected_combo[(row["Pclass"], row["Sex"])]
    assert (row["total"], row["supervivientes"]) == (total, survived)
    assert abs(row["tasa"] - survived / total) < 1e-6


shape: (6, 5)
┌────────┬────────┬───────┬────────────────┬──────────┐
│ Pclass ┆ Sex    ┆ total ┆ supervivientes ┆ tasa     │
│ ---    ┆ ---    ┆ ---   ┆ ---            ┆ ---      │
│ i64    ┆ enum   ┆ u32   ┆ i64            ┆ f64      │
╞════════╪════════╪═══════╪════════════════╪══════════╡
│ 1      ┆ female ┆ 94    ┆ 91             ┆ 0.968085 │
│ 1      ┆ male   ┆ 122   ┆ 45             ┆ 0.368852 │
│ 2      ┆ female ┆ 76    ┆ 70             ┆ 0.921053 │
│ 2      ┆ male   ┆ 108   ┆ 17             ┆ 0.157407 │
│ 3      ┆ female ┆ 144   ┆ 72             ┆ 0.5      │
│ 3      ┆ male   ┆ 347   ┆ 47             ┆ 0.135447 │
└────────┴────────┴───────┴────────────────┴──────────┘


## 6. Edad: menor de edad y adulta

Para este cálculo excluimos **solo** las filas cuyo `Age` es nulo, porque no pueden clasificarse como `<18` o `>=18`. No imputamos edades ni eliminamos filas del `df` original.

In [7]:
edad_clasificada = (
    df.filter(pl.col("Age").is_not_null())
    .with_columns(
        pl.when(pl.col("Age") < 18)
        .then(pl.lit("<18"))
        .otherwise(pl.lit(">=18"))
        .alias("grupo_edad")
    )
)
por_edad = resumen_supervivencia(edad_clasificada, ["grupo_edad"])
print(por_edad)
expected_age = {"<18": (113, 61), ">=18": (601, 229)}
for row in por_edad.iter_rows(named=True):
    total, survived = expected_age[row["grupo_edad"]]
    assert (row["total"], row["supervivientes"]) == (total, survived)
    assert abs(row["tasa"] - survived / total) < 1e-6
assert por_edad['total'].sum() == 714
assert por_sexo['Sex'].to_list() == ['female', 'male']
assert por_edad['grupo_edad'].to_list() == ['<18', '>=18']


shape: (2, 4)
┌────────────┬───────┬────────────────┬──────────┐
│ grupo_edad ┆ total ┆ supervivientes ┆ tasa     │
│ ---        ┆ ---   ┆ ---            ┆ ---      │
│ str        ┆ u32   ┆ i64            ┆ f64      │
╞════════════╪═══════╪════════════════╪══════════╡
│ <18        ┆ 113   ┆ 61             ┆ 0.539823 │
│ >=18       ┆ 601   ┆ 229            ┆ 0.381032 │
└────────────┴───────┴────────────────┴──────────┘


## 7. Lectura final

En estos datos, la supervivencia presenta asociaciones distintas según sexo, clase y grupo de edad: las tasas no son iguales entre categorías y también cambian al cruzar clase y sexo. Esta lectura es **descriptiva y asociacional**, no causal: el análisis no controla otras variables, no prueba mecanismos y no permite concluir que pertenecer a una categoría provoque el resultado.

El dataset original permanece intacto en `df`; únicamente se creó una tabla derivada para clasificar la edad no nula.

## 8. Pregunta propia del equipo experto

Formula una pregunta adicional, el filtro/agrupación y qué representa el denominador. Resuélvela con Polars sobre una tabla derivada, sin alterar ni imputar el CSV original. Devuelve total, supervivientes y tasa. No repitas una tabla del núcleo.

**Pregunta:** ¿cómo se asocia la supervivencia con el tamaño de la familia a bordo? Se define `tam_familia = SibSp + Parch + 1` (el propio pasajero más hermanos/cónyuge y padres/hijos) y se agrupa en tres categorías: `1 (solo)`, `2-4` y `5+`.

**Filtro, población y nulos:** no se filtra ninguna fila. `SibSp` y `Parch` no tienen nulos, así que el denominador de cada grupo son los pasajeros de ese grupo y la suma de los tres grupos es 891, el total del dataset. La tabla derivada `familia_clasificada` añade columnas a una copia; `df` no se modifica.

**Interpretación y limitación:** los pasajeros que viajan solos (537) sobreviven en un 30,4 %, los de familias de 2 a 4 personas (292) en un 57,9 % y los de 5 o más (62) en un 16,1 %. La tasa no crece ni decrece de forma lineal con el tamaño: es mayor en el grupo intermedio y baja en los extremos. Es una asociación descriptiva. El tamaño de la familia está relacionado con la clase, el sexo y la edad (por ejemplo, muchas de las familias grandes viajaban en tercera clase), y esta tabla no controla esas variables. Además, el grupo `5+` es pequeño (62 pasajeros), así que su tasa es poco estable, y `SibSp + Parch + 1` solo cuenta familiares a bordo, no el tipo de relación.

Después prepara la entrega experta según [entrega-comparativa-motores-titanic.md](entrega-comparativa-motores-titanic.md).

In [8]:
familia_clasificada = df.with_columns(
    (pl.col("SibSp") + pl.col("Parch") + 1).alias("tam_familia")
).with_columns(
    pl.when(pl.col("tam_familia") == 1).then(pl.lit("1 (solo)"))
    .when(pl.col("tam_familia") <= 4).then(pl.lit("2-4"))
    .otherwise(pl.lit("5+"))
    .cast(pl.Enum(["1 (solo)", "2-4", "5+"]))
    .alias("grupo_familia")
)

# Denominador: pasajeros de cada grupo. No hay nulos en SibSp ni Parch, así que no se excluye ninguna fila.
por_familia = resumen_supervivencia(familia_clasificada, ["grupo_familia"])
print(por_familia)

expected_family = {"1 (solo)": (537, 163), "2-4": (292, 169), "5+": (62, 10)}
for row in por_familia.iter_rows(named=True):
    total, survived = expected_family[row["grupo_familia"]]
    assert (row["total"], row["supervivientes"]) == (total, survived)
    assert abs(row["tasa"] - survived / total) < 1e-6
assert por_familia["total"].sum() == 891
assert por_familia["supervivientes"].sum() == 342
assert por_familia["grupo_familia"].cast(pl.Utf8).to_list() == ["1 (solo)", "2-4", "5+"]
assert df.shape == (891, 12)  # el dataset original no se ha modificado

shape: (3, 4)
┌───────────────┬───────┬────────────────┬──────────┐
│ grupo_familia ┆ total ┆ supervivientes ┆ tasa     │
│ ---           ┆ ---   ┆ ---            ┆ ---      │
│ enum          ┆ u32   ┆ i64            ┆ f64      │
╞═══════════════╪═══════╪════════════════╪══════════╡
│ 1 (solo)      ┆ 537   ┆ 163            ┆ 0.303538 │
│ 2-4           ┆ 292   ┆ 169            ┆ 0.578767 │
│ 5+            ┆ 62    ┆ 10             ┆ 0.16129  │
└───────────────┴───────┴────────────────┴──────────┘
